In [1]:
#!pip install torchvision
#!pip install facenet_pytorch

import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
# from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from VideoSwinTransformer import SwinTransformer3D
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+VST + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()


In [2]:

logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-21 01:23:45,406]::3827333478::DEBUG::=============r2plus1d+VST + AST fusion after transformer blocks+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [5]:
validation_set_data=getdata("/data1/qx/mean_based_arg/KETI/data/scene3_valid_set30s.dat","valid")
print(1)
train_set_data=getdata("/data1/qx/mean_based_arg/KETI/data/scene3_train_set30s.dat","train")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)/120
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)

1


In [9]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[3],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[3],transform=transform)
train_dataloader = DataLoader(dataset=train_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
val_dataloader = DataLoader(dataset=val_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
max_value=0
print(1)

1


In [10]:
model=SwinTransformer3D()#num_frames=frame_nums,batchsz = batchsz
model.to(device)
criterion = torch.nn.L1Loss().to(device)  # 손실함수
optimizer = torch.optim.AdamW(model.parameters(), lr=lr) # 옵티마이저

if test_flage==True:
    checkpoint=torch.load(save_model_file_path.format('model',start_epoch,'pth'), map_location=device)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    # criterion.load_state_dict(checkpoint["loss"])
train_avg_loss0=[]
val_avg_loss0=[]
print(1)

1


In [11]:
from torchinfo import summary

summary(model, input_size = ((4, 3,15, 224, 224)), col_names = ['input_size','output_size','num_params'])

Layer (type:depth-idx)                             Input Shape               Output Shape              Param #
SwinTransformer3D                                  [4, 3, 15, 224, 224]      [4, 5]                    --
├─VideoResNet: 1-1                                 [4, 3, 15, 224, 224]      [4, 512, 2, 14, 14]       205,200
│    └─R2Plus1dStem: 2-1                           [4, 3, 15, 224, 224]      [4, 64, 15, 112, 112]     --
│    │    └─Conv3d: 3-1                            [4, 3, 15, 224, 224]      [4, 45, 15, 112, 112]     6,615
│    │    └─BatchNorm3d: 3-2                       [4, 45, 15, 112, 112]     [4, 45, 15, 112, 112]     90
│    │    └─ReLU: 3-3                              [4, 45, 15, 112, 112]     [4, 45, 15, 112, 112]     --
│    │    └─Conv3d: 3-4                            [4, 45, 15, 112, 112]     [4, 64, 15, 112, 112]     8,640
│    │    └─BatchNorm3d: 3-5                       [4, 64, 15, 112, 112]     [4, 64, 15, 112, 112]     128
│    │    └─ReLU: 3-6        

In [12]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [13]:
# ============================== Multi-seed training ==============================
import random
import numpy as np
import torch
from torch.utils.data import DataLoader

SEEDS = [42, 123, 777, 1004, 2024]
PATIENCE = 20


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


all_seed_results = []

for seed in SEEDS:
    print("\n" + "=" * 80)
    print(f"Training seed: {seed}")
    print("=" * 80)

    # Each seed starts from a completely fresh random state/model/optimizer.
    set_seed(seed)

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_dataloader = DataLoader(
        dataset=train_set_data,
        batch_size=batchsz,
        shuffle=True,
        num_workers=num_workerssz,
        worker_init_fn=seed_worker,
        generator=generator,
    )
    val_dataloader = DataLoader(
        dataset=val_set_data,
        batch_size=batchsz,
        shuffle=False,
        num_workers=num_workerssz,
        worker_init_fn=seed_worker,
    )

    model = SwinTransformer3D().to(device)
    criterion = torch.nn.L1Loss().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    best_mae = float('inf')
    best_epoch = 0
    patience_counter = 0
    best_model_path = f'./model_best_seed_{seed}.pth'

    for epoch in range(1, epochs + 1):
        # ------------------------------ train ------------------------------
        model.train()
        train_avg_loss = 0.0

        for audiodata, fullshot, big_five_data in tqdm(train_dataloader):
            fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
            big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
            audiodata = audiodata.to(device)  # kept to preserve the original pipeline

            optimizer.zero_grad()
            hypothesis = model(fullshot)
            loss = criterion(hypothesis, big_five_data)
            loss.backward()
            optimizer.step()
            train_avg_loss += loss.item()

        train_avg_loss /= len(train_dataloader)

        # ---------------------------- validation ---------------------------
        model.eval()
        val_avg_loss = 0.0
        val_avg_o_loss = 0.0
        val_avg_c_loss = 0.0
        val_avg_e_loss = 0.0
        val_avg_a_loss = 0.0
        val_avg_n_loss = 0.0

        with torch.no_grad():
            for audiodata, fullshot, big_five_data in tqdm(val_dataloader):
                fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
                big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
                audiodata = audiodata.to(device)

                hypothesis = model(fullshot)
                val_loss = criterion(hypothesis, big_five_data)

                hypothesiso, hypothesisc, hypothesise, hypothesisa, hypothesisn = extract_ocean(hypothesis, len(fullshot))
                big_five_datao, big_five_datac, big_five_datae, big_five_dataa, big_five_datan = extract_ocean(big_five_data, len(fullshot))

                # extract_ocean returns CPU tensors, so use a CPU L1 calculation here.
                val_avg_o_loss += torch.nn.functional.l1_loss(hypothesiso, big_five_datao).item()
                val_avg_c_loss += torch.nn.functional.l1_loss(hypothesisc, big_five_datac).item()
                val_avg_e_loss += torch.nn.functional.l1_loss(hypothesise, big_five_datae).item()
                val_avg_a_loss += torch.nn.functional.l1_loss(hypothesisa, big_five_dataa).item()
                val_avg_n_loss += torch.nn.functional.l1_loss(hypothesisn, big_five_datan).item()
                val_avg_loss += val_loss.item()

        n_val_batches = len(val_dataloader)
        val_avg_loss /= n_val_batches
        val_avg_o_loss /= n_val_batches
        val_avg_c_loss /= n_val_batches
        val_avg_e_loss /= n_val_batches
        val_avg_a_loss /= n_val_batches
        val_avg_n_loss /= n_val_batches

        current_1_mae = 1.0 - val_avg_loss

        # ----------------------- checkpoint / patience ----------------------
        if val_avg_loss < best_mae:
            best_mae = val_avg_loss
            best_epoch = epoch
            patience_counter = 0
            torch.save({
                'seed': seed,
                'epoch': epoch,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'MAE': val_avg_loss,
                '1-MAE': current_1_mae,
            }, best_model_path)
        else:
            patience_counter += 1

        best_1_mae = 1.0 - best_mae

        # Every epoch prints current score + best-so-far + patience.
        print(
            f"[Seed {seed}] Epoch {epoch:03d}/{epochs} | "
            f"Train 1-MAE: {1-train_avg_loss:.6f} | "
            f"Val 1-MAE: {current_1_mae:.6f} | "
            f"Best 1-MAE: {best_1_mae:.6f} (epoch {best_epoch}) | "
            f"Patience: {patience_counter}/{PATIENCE}"
        )
        print(
            f"           O: {1-val_avg_o_loss:.6f} | C: {1-val_avg_c_loss:.6f} | "
            f"E: {1-val_avg_e_loss:.6f} | A: {1-val_avg_a_loss:.6f} | N: {1-val_avg_n_loss:.6f}"
        )

        logger.debug(
            'Seed: %s, Epoch: %s, Val 1-MAE: %.6f, Best 1-MAE: %.6f, Best epoch: %s, Patience: %s/%s',
            seed, epoch, current_1_mae, best_1_mae, best_epoch, patience_counter, PATIENCE
        )

        if patience_counter >= PATIENCE:
            print(
                f"Early stopping seed {seed} at epoch {epoch}. "
                f"Best epoch = {best_epoch}, Best 1-MAE = {best_1_mae:.6f}"
            )
            break

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    all_seed_results.append({
        'seed': seed,
        'best_epoch': best_epoch,
        'MAE': best_mae,
        '1-MAE': 1.0 - best_mae,
        'checkpoint': best_model_path,
    })

    print(
        f"\nSeed {seed} finished | Best epoch = {best_epoch} | "
        f"Best MAE = {best_mae:.6f} | Best 1-MAE = {1-best_mae:.6f}"
    )

    del model, optimizer, criterion
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================== Final summary ==============================
maes = np.array([r['MAE'] for r in all_seed_results], dtype=float)
scores = np.array([r['1-MAE'] for r in all_seed_results], dtype=float)

print("\n" + "=" * 80)
print(f"FINAL {len(SEEDS)}-SEED RESULTS")
print("=" * 80)
for r in all_seed_results:
    print(
        f"Seed {r['seed']:4d} | Best epoch {r['best_epoch']:3d} | "
        f"MAE = {r['MAE']:.6f} | 1-MAE = {r['1-MAE']:.6f}"
    )
print("-" * 80)
print(f"MAE   : {maes.mean():.6f} ± {maes.std(ddof=1):.6f}")
print(f"1-MAE : {scores.mean():.6f} ± {scores.std(ddof=1):.6f}")



Training seed: 42


100%|██████████| 337/337 [00:29<00:00, 11.57it/s]
[2026-09-21 01:28:30,273]::980139715::DEBUG::Seed: 42, Epoch: 1, Val 1-MAE: 0.885292, Best 1-MAE: 0.885292, Best epoch: 1, Patience: 0/20


[Seed 42] Epoch 001/120 | Train 1-MAE: 0.878105 | Val 1-MAE: 0.885292 | Best 1-MAE: 0.885292 (epoch 1) | Patience: 0/20
           O: 0.908466 | C: 0.886239 | E: 0.880784 | A: 0.893394 | N: 0.857577


100%|██████████| 337/337 [00:29<00:00, 11.47it/s]
[2026-09-21 01:33:08,181]::980139715::DEBUG::Seed: 42, Epoch: 2, Val 1-MAE: 0.906768, Best 1-MAE: 0.906768, Best epoch: 2, Patience: 0/20


[Seed 42] Epoch 002/120 | Train 1-MAE: 0.912232 | Val 1-MAE: 0.906768 | Best 1-MAE: 0.906768 (epoch 2) | Patience: 0/20
           O: 0.907581 | C: 0.907657 | E: 0.885839 | A: 0.934821 | N: 0.897940


...

[Seed 2024] Epoch 053/120 | Train 1-MAE: 0.992372 | Val 1-MAE: 0.907636 | Best 1-MAE: 0.909479 (epoch 35) | Patience: 18/20
           O: 0.908992 | C: 0.913644 | E: 0.880666 | A: 0.931304 | N: 0.903571


100%|██████████| 337/337 [00:29<00:00, 11.52it/s]
[2026-09-21 16:55:24,032]::980139715::DEBUG::Seed: 2024, Epoch: 54, Val 1-MAE: 0.906530, Best 1-MAE: 0.909479, Best epoch: 35, Patience: 19/20


[Seed 2024] Epoch 054/120 | Train 1-MAE: 0.992123 | Val 1-MAE: 0.906530 | Best 1-MAE: 0.909479 (epoch 35) | Patience: 19/20
           O: 0.908029 | C: 0.910797 | E: 0.879171 | A: 0.930944 | N: 0.903709


100%|██████████| 337/337 [00:29<00:00, 11.40it/s]
[2026-09-21 17:00:01,756]::980139715::DEBUG::Seed: 2024, Epoch: 55, Val 1-MAE: 0.906596, Best 1-MAE: 0.909479, Best epoch: 35, Patience: 20/20


[Seed 2024] Epoch 055/120 | Train 1-MAE: 0.992177 | Val 1-MAE: 0.906596 | Best 1-MAE: 0.909479 (epoch 35) | Patience: 20/20
           O: 0.909922 | C: 0.911008 | E: 0.881520 | A: 0.930031 | N: 0.900497
Early stopping seed 2024 at epoch 55. Best epoch = 35, Best 1-MAE = 0.909479

Seed 2024 finished | Best epoch = 35 | Best MAE = 0.090521 | Best 1-MAE = 0.909479

FINAL 5-SEED RESULTS
Seed   42 | Best epoch  22 | MAE = 0.092959 | 1-MAE = 0.907041
Seed  123 | Best epoch   3 | MAE = 0.092118 | 1-MAE = 0.907882
Seed  777 | Best epoch  27 | MAE = 0.091339 | 1-MAE = 0.908661
Seed 1004 | Best epoch  15 | MAE = 0.090173 | 1-MAE = 0.909827
Seed 2024 | Best epoch  35 | MAE = 0.090521 | 1-MAE = 0.909479
--------------------------------------------------------------------------------
MAE   : 0.091422 ± 0.001143
1-MAE : 0.908578 ± 0.001143


In [14]:
# from torchinfo import summary

# summary(model, input_size = (4,15,4,3,224,224))